# IMPORTING

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import joblib

In [2]:
train = pd.read_csv(r'C:\Users\merug\Desktop\Taxi Trip distance prediction\taxi trip distance prediction\Raw Dataset\train.csv')
test  = pd.read_csv(r'C:\Users\merug\Desktop\Taxi Trip distance prediction\taxi trip distance prediction\Raw Dataset\Test.csv')

train['is_train'] = 1
test['is_train'] = 0

df = pd.concat([train, test], axis=0).reset_index(drop=True)

# Feature Engineering

In [3]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], format='%m/%d/%y %H:%M', errors='coerce')
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'], format='%m/%d/%y %H:%M', errors='coerce')

df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['month'] = df['pickup_datetime'].dt.month

df['traffic_congestion_level'] = df['traffic_congestion_level'].map({
    'low': 1,
    'medium': 2,
    'high': 3
})


pickup_dummies = pd.get_dummies(df['pickup_borough'], prefix='pickup')
dropoff_dummies = pd.get_dummies(df['dropoff_borough'], prefix='dropoff')

df = pd.concat([df, pickup_dummies, dropoff_dummies], axis=1)

df.drop(['pickup_borough', 'dropoff_borough'], axis=1, inplace=True)

In [4]:

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['haversine_distance'] = haversine(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

df['lat_diff'] = df['dropoff_latitude'] - df['pickup_latitude']
df['lon_diff'] = df['dropoff_longitude'] - df['pickup_longitude']

df['trip_duration'] = (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds()
df = df[df['trip_duration'] > 0]

df['avg_speed'] = df['haversine_distance'] / (df['trip_duration'] + 1)

df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
df['rush_hour'] = df['hour'].isin([8,9,17,18,19]).astype(int)

df.drop(['pickup_datetime', 'dropoff_datetime'], axis=1, inplace=True)

In [5]:
train_df = df[df['is_train'] == 1].drop('is_train', axis=1)
test_df  = df[df['is_train'] == 0].drop('is_train', axis=1)

# Data Spliting 

In [6]:

features = [
    'pickup_latitude', 'pickup_longitude',
    'dropoff_latitude', 'dropoff_longitude',
    'passenger_count',
    'traffic_congestion_level',
    'haversine_distance','rush_hour','day_of_week','avg_speed'
]

X = train_df[features]
y = train_df['trip_distance_miles']

test_X = test_df[features]

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Selection

In [8]:
cat = CatBoostRegressor(
    iterations=1800,
    learning_rate=0.025,
    depth=6,
    verbose=200,
    random_seed=42
)

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb = LGBMRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [9]:
cat.fit(X_train, y_train)
xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)

0:	learn: 7.7068085	total: 153ms	remaining: 4m 35s
200:	learn: 0.8094595	total: 1.33s	remaining: 10.6s
400:	learn: 0.7813434	total: 2.47s	remaining: 8.62s
600:	learn: 0.7668493	total: 3.84s	remaining: 7.65s
800:	learn: 0.7524048	total: 4.97s	remaining: 6.2s
1000:	learn: 0.7393431	total: 6.09s	remaining: 4.86s
1200:	learn: 0.7272863	total: 7.26s	remaining: 3.62s
1400:	learn: 0.7162214	total: 8.43s	remaining: 2.4s
1600:	learn: 0.7057066	total: 9.44s	remaining: 1.17s
1799:	learn: 0.6956515	total: 10.4s	remaining: 0us
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001330 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 11759, number of used features: 10
[LightGBM] [Info] Start training from score 13.940122


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, n_estimators=1200,
              random_state=42, subsample=0.8)

In [10]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

cat_pred = cat.predict(X_val)
xgb_pred = xgb.predict(X_val)
lgb_pred = lgb.predict(X_val)

print("Cat RMSE:", rmse(y_val, cat_pred))
print("XGB RMSE:", rmse(y_val, xgb_pred))
print("LGB RMSE:", rmse(y_val, lgb_pred))

Cat RMSE: 0.7768563387059666
XGB RMSE: 0.7817494285026763
LGB RMSE: 0.7918853672807314


# Ensamble method

In [11]:
val_preds = (
    0.6 * cat_pred +
    0.4 * xgb_pred +
    0.0 * lgb_pred
)

print("Ensemble RMSE:", rmse(y_val, val_preds))

Ensemble RMSE: 0.7742015010667088


In [12]:
final_preds = (
    0.6 * cat.predict(test_X) +
    0.4 * xgb.predict(test_X) +
    0.0 * lgb.predict(test_X)
)

print(final_preds[:10])

[20.63485987  7.45697158 24.30667397 21.97082397 19.16796735 14.32679705
 10.01664261 11.99500837  7.17795621 20.65175938]


In [13]:

train_pred = cat.predict(X_train)


val_pred = cat.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

print("Training RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)

Training RMSE: 0.6956514647529121
Validation RMSE: 0.7768563387059666


In [14]:
import joblib

joblib.dump(cat, r"C:\Users\merug\Desktop\Taxi Trip distance prediction\taxi trip distance prediction\model files\cat_model.pkl")
joblib.dump(xgb, r"C:\Users\merug\Desktop\Taxi Trip distance prediction\taxi trip distance prediction\model files\xgb_model.pkl")
joblib.dump(lgb, r"C:\Users\merug\Desktop\Taxi Trip distance prediction\taxi trip distance prediction\model files\lgb_model.pkl")

['C:\\Users\\merug\\Desktop\\Taxi Trip distance prediction\\taxi trip distance prediction\\model files\\lgb_model.pkl']